# **02477 | Exam 2026 — Prediction Notebook**

<div class="alert alert-block alert-danger">

### Pre-exam predictions based on rotation across 2024, 2024R, 2025, 2025R

Each section labelled with predicted confidence. Read once, then close and sleep.

</div>



---

## 🔴 Part 1 — Linear Gaussian System `CONFIDENCE: HIGH`

**Appeared in:** 2024, 2025R. Almost certain to appear again.

**What to expect:** A chain $z = Ax + b + n_1$, $y = Wz + n_2$ (or similar). Asked for $p(z|x)$, $p(x|z)$, $p(y)$.

**The three-step recipe:**

**Step 1 — Conditional** $p(z|x)$: condition on $x$ → it becomes a constant:
$$p(z|x) = \mathcal{N}(z \mid Ax + b,\ \text{Cov}(n_1))$$

**Step 2 — Posterior** $p(x|z)$: use the linear Gaussian system formula. Match $W \to A$, $b \to b$, $\Sigma_y \to \text{Cov}(n_1)$, $\Sigma_z \to \text{Cov}(x)$:
$$\Sigma_{x|z}^{-1} = \Sigma_x^{-1} + A^T \Sigma_{n_1}^{-1} A$$
$$\mu_{x|z} = \Sigma_{x|z}\left[A^T \Sigma_{n_1}^{-1}(z - b) + \Sigma_x^{-1}\mu_x\right]$$

**Step 3 — Marginal** $p(y)$: propagate through both equations using the marginalisation identity:
$$\text{Cov}(z) = A\,\text{Cov}(x)\,A^T + \text{Cov}(n_1)$$
$$p(y) = \mathcal{N}(y \mid W\mu_z + b_2,\ \text{Cov}(n_2) + W\,\text{Cov}(z)\,W^T)$$

<span style="color: red;">

**The one rule:** when you condition on something, it becomes a constant. $\text{Cov}(z|x) = \text{Cov}(n_1)$ only — not $A\,\text{Cov}(x)\,A^T$ because $x$ is fixed.

</span>



---

## 🔴 Part 2 — Bayesian Linear Regression or GP Regression `CONFIDENCE: HIGH`

**Appeared in:** 2024 (GP), 2024R (GP), 2025 (linear). Due to rotate to one of these.

### Case A — Bayesian Linear Regression

$$S = (\alpha I + \beta \Phi^T\Phi)^{-1}, \quad m = \beta S \Phi^T y$$
$$p(f^*|y,x^*) = \mathcal{N}(f^* \mid m^T\phi^*,\ \phi^{*T} S \phi^*)$$
$$p(y^*|y,x^*) = \mathcal{N}(y^* \mid m^T\phi^*,\ \phi^{*T} S \phi^* + \sigma^2)$$

```python
beta = 1/sigma2
alpha = 1/tau2   # prior precision
S = jnp.linalg.inv(alpha*jnp.eye(D) + beta*Phi.T@Phi)
m = beta * S @ Phi.T @ y
phi_star = design_matrix(jnp.array([x_star])).ravel()
mu_pred  = m @ phi_star
var_pred = phi_star @ S @ phi_star + sigma2
```

### Case B — GP Regression

$$\mu_{f^*|y} = k_*(K+\sigma^2 I)^{-1}y$$
$$\sigma^2_{f^*|y} = k(x^*,x^*) - k_*(K+\sigma^2 I)^{-1}k_*^T$$

```python
k_star = kernel(x_star, x_train)          # shape (N,)
K      = kernel(x_train[:,None], x_train[None,:])
mu     = k_star @ jnp.linalg.solve(K + sigma2*jnp.eye(N), y)
var    = kernel(x_star, x_star) - k_star @ jnp.linalg.solve(K + sigma2*jnp.eye(N), k_star)
```

---



## 🟠 Part 3 — Mixture Model + Variational Inference `CONFIDENCE: MEDIUM-HIGH`

**Appeared in:** 2025 (mixture + VI), 2025R (mixture). Strong pattern.

**What to expect:** A latent binary $z_n \in \{0,1\}$, marginalize out $z$ (sum rule), marginalize out $\theta$ (integral), then entropy/credibility/predictive from the variational approximation.

**Marginalise out discrete $z$** (sum rule):
$$p(y, \theta) = \sum_{z \in \{0,1\}} p(y|\theta, z)\,p(z)\,p(\theta)$$
$$= (1-\pi)\mathcal{N}(y|0,1)\mathcal{N}(\theta|0,\alpha^{-1}I) + \pi\,\mathcal{N}(y|\theta^Tx, \sigma^2)\mathcal{N}(\theta|0,\alpha^{-1}I)$$

**Marginalise out continuous $\theta$** (marginalisation identity):
$$\int \mathcal{N}(y|\theta^Tx, \sigma^2)\mathcal{N}(\theta|0,\alpha^{-1}I)\,d\theta = \mathcal{N}(y \mid 0,\ \sigma^2 + \alpha^{-1}x^Tx)$$

**Entropy of mean-field Gaussian** $q(\theta) = \prod_i \mathcal{N}(\theta_i|m_i, v_i)$:
$$H[q] = \sum_i \frac{1}{2}\ln(2\pi e\, v_i)$$

```python
H = jnp.sum(0.5 * jnp.log(2 * jnp.pi * jnp.e * v))
```

**Posterior predictive using $q^*(\theta)$** — same marginalisation pattern:
$$p(y^*|y,x^*) = \mathcal{N}(y^*|0,1) + \mathcal{N}(y^* \mid m^Tx^*,\ \sigma^2 + x^{*T}Sx^*)$$

<span style="color: red;">

**Trap:** the result is a **sum of two Gaussians** (a mixture), not a single Gaussian. Do not collapse it.

**KL and variational families:** if $Q_1 \subset Q_2$, then $\text{KL}[q_2^*\|p] \leq \text{KL}[q_1^*\|p]$. Larger family = smaller or equal KL. Always.

</span>

---



## 🟡 Part 4 — GP Classification or Poisson GLM `CONFIDENCE: MEDIUM`

### Case A — GP Classification (appeared 2025)

**Step 1:** Laplace approximation — given $m = \hat{f}$ and $S$:
$$q(f) = \mathcal{N}(f \mid m, S), \quad S = (K^{-1} + \Lambda)^{-1}$$

**Step 2:** Posterior predictive for $f^*$:
$$\mu_{f^*} = k_*^T K^{-1} m, \quad \sigma^2_{f^*} = k(x^*,x^*) - k_*^T(K+\Lambda^{-1})^{-1}k_*$$

**Step 3:** Probit approximation:
$$p(y^*=1|y,x^*) \approx \Phi\!\left(\frac{\mu_{f^*}}{\sqrt{\frac{8}{\pi} + \sigma^2_{f^*}}}\right)$$

```python
from scipy.stats import norm
p = norm.cdf(mu_fstar / jnp.sqrt(8/jnp.pi + var_fstar))   # norm.CDF not pdf!
```

**Prior predictive** (no data): always $0.5$ when prior mean is zero.



### 🔴 Case B — Poisson GLM (SURPRISE — never appeared in May exam) `CONFIDENCE: MEDIUM`

**Model:**
$$y_n | x_n, w \sim \text{Poisson}(\mu_n), \quad \mu_n = \exp(w^T x_n), \quad w \sim \mathcal{N}(0, \alpha^{-1}I)$$

**Log-posterior** (what you pass to MCMC):
$$\log p(w|y) \propto \sum_n \left[y_n w^T x_n - \exp(w^T x_n)\right] - \frac{\alpha}{2}w^Tw$$

```python
def log_posterior(w):
    mu = jnp.exp(x @ w)                        # shape (N,)
    log_lik = jnp.sum(y * jnp.log(mu) - mu)   # drop log(y!) constant
    log_prior = -0.5 * alpha * jnp.dot(w, w)
    return log_lik + log_prior
```

**Prior mean of $\mu(x^*)$** — common sub-question:
$$\mathbb{E}_{p(w)}[\exp(w^T x^*)] = \exp\!\left(\frac{\|x^*\|^2}{2\alpha}\right)$$
> **Special case:** if $x^* = 0$, then $\mu(x^*) = \exp(\text{constant})$ — not random at all.

**Posterior predictive** (two-layer sampling):
```python
# Step 1: samples from posterior (MCMC or Laplace)
mu_star = jnp.exp(samples @ x_star)             # rate for each sample
# Step 2: sample y* from Poisson
y_star = np.random.poisson(np.array(mu_star))   # needs numpy!
# Step 3: credibility interval
jnp.percentile(jnp.array(y_star), jnp.array([5., 95.]))  # 90% CI
```

<span style="color: red;">

**Trap:** $\mu^* = \exp(w^Tx^*)$ is the **rate**, not $y^*$. Always sample $y^* \sim \text{Poisson}(\mu^*)$ as a second step. Use `np.random.poisson` not JAX.

</span>

---



## 🟡 Part 5 — MCMC on Non-linear Model `CONFIDENCE: MEDIUM`

**Appeared in:** 2025 (latent variable model), 2025R (tanh regression). Strong pattern for Part 5.

**Universal log_posterior template:**
```python
def log_posterior(theta):
    # Step 1: evaluate the model
    f = theta[1] * jnp.tanh(theta[0] * x_obs)  # or whatever f(x|theta) is
    # Step 2: log likelihood
    log_lik = log_npdf(y_obs, f, sigma2)         # Gaussian: one obs
    # Step 3: log prior — sum over each scalar parameter
    log_prior = log_npdf(theta[0], 0, 1) + log_npdf(theta[1], 0, 1)
    return log_lik + log_prior

samples = metropolis(log_posterior, num_params=2, tau=jnp.sqrt(2),
                     num_iter=10**4, theta_init=jnp.array([0., 0.]), seed=0)
warmup = int(0.1 * len(samples))
samples = samples[warmup:]
```

**Common downstream questions from samples:**
```python
# P(theta_1 > 0)
jnp.mean(samples[:,0] > 0)

# E[f(x*)] for non-linear f
f_star = samples[:,1] * jnp.tanh(samples[:,0] * x_star)
jnp.mean(f_star)

# P(f(x*) > c)
jnp.mean(f_star > c)

# 95% CI for y* (add observation noise!)
eps = jnp.array(np.random.normal(0, jnp.sqrt(sigma2), len(f_star)))
y_star = f_star + eps
jnp.percentile(y_star, jnp.array([2.5, 97.5]))

# R-hat convergence (two chains)
W = (jnp.var(chain1) + jnp.var(chain2)) / 2
B = len(chain1) * jnp.var(jnp.array([jnp.mean(chain1), jnp.mean(chain2)]))
R_hat = jnp.sqrt((len(chain1)-1)/len(chain1) + B/(len(chain1)*W))
# converged if R_hat < 1.1
```

<span style="color: red;">

**The three MCMC traps:**
1. Use `tau` from the question — don't guess it
2. `log_prior` for vector $w$: sum `log_npdf` over each component separately, or use `mvn.logpdf`
3. $f^*$ vs $y^*$: add $\epsilon \sim \mathcal{N}(0,\sigma^2)$ before taking percentiles if asked for $p(y^*|y,x^*)$

</span>



---

## 🔵 Wildcard — Multi-class Classification `CONFIDENCE: LOW`

**Only appeared in 2024R.** Low probability but worth one glance.

$$p(y=k|x) = \frac{\exp(a_k)}{\sum_j \exp(a_j)} = \text{softmax}(a)_k, \quad a_k = w_k^T x$$

Posterior predictive via sampling:
```python
f_star = w_samples @ x_star          # shape (S, K)
probs  = jax.nn.softmax(f_star, axis=1)
p_class_k = jnp.mean(probs[:, k])
```

---

<div class="alert alert-block alert-warning">

### Last-minute checklist for tomorrow

</div>

Before writing any answer:
1. **Underline what you're conditioning on** — it becomes a constant
2. **Check the domain of $y$** — real → Gaussian, binary → Bernoulli, count → Poisson
3. **$f^*$ vs $y^*$** — add $\sigma^2$ / sample from likelihood if observation is asked
4. **Probit uses `norm.cdf`** — never `norm.pdf`
5. **`norm` takes std dev** $\sqrt{v}$, not variance $v$
6. **Gradient at MAP = 0** — one line, no derivation needed
7. **Report 2 decimal places** — always
```
```

## 🔵 Wildcard — Poisson GLM `CONFIDENCE: LOW-MEDIUM`

**Only appeared in 2024R Part 4.** Overdue for a May appearance.

Already covered fully in Part 4 Case B above. Key things to remember:
- `log_lik = jnp.sum(y * jnp.log(mu) - mu)` — always `.sum()`
- Prior mean of $\mu(x^*)$: use MGF formula $\exp(\|x^*\|^2 / 2\alpha)$
- Two-layer sampling: first $\mu^* = \exp(w^Tx^*)$, then `np.random.poisson(mu_star)`

---



## 🔵 Wildcard — R-hat / Convergence Diagnostics `CONFIDENCE: LOW-MEDIUM`

**Appeared in 2025R implicitly.** Could appear as a standalone question after MCMC.

```python
S = len(chain1)
W = (jnp.var(chain1) + jnp.var(chain2)) / 2
grand_mean = (jnp.mean(chain1) + jnp.mean(chain2)) / 2
B = S * ((jnp.mean(chain1) - grand_mean)**2 + 
         (jnp.mean(chain2) - grand_mean)**2)
R_hat = jnp.sqrt((S-1)/S + B/(S*W))
# converged if R_hat < 1.1
```

---



## 🔵 Wildcard — Heteroscedastic Model `CONFIDENCE: LOW-MEDIUM`

**Only appeared in 2025R Part 4.** The model where variance depends on input:
$$p(y_n|w,x_n) = \mathcal{N}(y_n \mid w_1 + w_2 x_n,\ \exp(w_3 + w_4 x_n))$$

**Entropy** of mean-field $q(w) = \prod_i \mathcal{N}(w_i|m_i,v_i)$:
$$H[q] = \sum_i \frac{1}{2}\ln(2\pi e\, v_i)$$

**Monte Carlo predictive:**
```python
means = w_samples[:,0] + w_samples[:,1] * x_star
vars_ = jnp.exp(w_samples[:,2] + w_samples[:,3] * x_star)
likelihoods = norm.pdf(y_star, loc=means, scale=jnp.sqrt(vars_))
p_pred = jnp.mean(likelihoods)
```

---



## 🔵 Wildcard — ELBO / BBVI `CONFIDENCE: LOW`

**Appeared in 2024 Part 4.** Could reappear as a VI question without mixture context.

$$\mathcal{L} = \mathbb{E}_q[\ln p(y,w)] + H[q]$$
$$= \mathbb{E}_q[\ln p(y|w)] - \text{KL}[q(w)\|p(w)]$$

**KL between two Gaussians** (mean-field, each component):
$$\text{KL}[\mathcal{N}(m,v)\|\mathcal{N}(0,\alpha^{-1})] = \frac{1}{2}\left(\alpha v + \alpha m^2 - 1 - \ln(\alpha v)\right)$$

**Expected log-likelihood** — always expand $(y - f(w))^2$:
$$\mathbb{E}_q[(y - f(w))^2] = y^2 - 2y\,\mathbb{E}_q[f] + \mathbb{E}_q[f^2]$$

For bilinear $f = w_1 w_2$, mean-field:
$$\mathbb{E}_q[f] = m_1 m_2, \quad \mathbb{E}_q[f^2] = (m_1^2+v_1)(m_2^2+v_2)$$

---



## 🔵 Wildcard — Decision Theory `CONFIDENCE: LOW`

**Appeared in 2024R briefly.** One question, easy marks if you know the formula.

$$\hat{y}^* = \arg\max_{\hat{y}} \sum_k p(y^*=k|y,x^*) \cdot U(k, \hat{y})$$

For 0/1 loss: just predict the most probable class.

```python
EU_0 = p0 * U[0,0] + p1 * U[1,0]   # expected utility predicting 0
EU_1 = p0 * U[0,1] + p1 * U[1,1]   # expected utility predicting 1
decision = jnp.argmax(jnp.array([EU_0, EU_1]))
```

---



## 🔵 Wildcard — Marginal Likelihood / Model Selection `CONFIDENCE: LOW`

**Appeared in 2025 Part 1 implicitly.** Could appear as "which model is better?"

For Bayesian linear regression:
$$\log p(y|\alpha,\beta) = -\frac{N}{2}\ln(2\pi) - \frac{1}{2}\ln|\sigma^2 I + \alpha^{-1}\Phi\Phi^T| - \frac{1}{2}y^T(\sigma^2 I + \alpha^{-1}\Phi\Phi^T)^{-1}y$$

The model with **higher** log marginal likelihood wins. No threshold needed — just compare.